<a href="https://colab.research.google.com/github/jmarrietar/LLMs-from-scratch-study/blob/feature%2Fstudy-session/Chapter_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-requisites

In [ ]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads

        self.head_dim = d_out // num_heads  # -> Reduces de projection dum to match desired output dim

        # Seteamos las Matrices de pesos GRANDES
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Una Layer Linear para combinar el output de los heads.
        self.out_proj = nn.Linear(d_out, d_out)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )

    def forward(self, x):

        b, num_tokens, d_in = x.shape

        # La multiplicacion de los pesos Grandes con el Input
        # Esto nos da unas matrices (Q,K,V) grandes tambien

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Truco: Aca implicitamente SEPARO La matriz agregandole num_heads a la dimension
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)


        # Transpose from shape(b, num_tokens, num_heads, head_dim) to (b,num_heads, num_tokens,head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # Ahora: Todo como normalmente se estaba haciendo (Calcular attn scores)
        attn_scores = queries @ keys.transpose(2,3) # Dot product for each head

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # MASK: Aplicarle mascara para hacerlo Causal Attention
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Attention weigth -> Apply softmax + escalamiento
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # Aplicar Dropout
        attn_weights = self.dropout(attn_weights)

        # Obtener Vector de Contexto (Aplicarlo ahora si )
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # combinar heads

        # optional projection

        context_vec = self.out_proj(context_vec)

        return context_vec

# LLM Architecture

Ya como tenemos *entendido* el Multi Head Attention. Lo que viene ahora es construir los componentes de la LLM.

Nota: `gpt-2` tiene 124 millones de parametros. `gpt-3` es lo mismo solo que escalaron a 1.5 Billones de parametros.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # Vocabulary size
    "context_length" : 1024, # Context length (El maximo numero de input tokens el modelo puede soportar con embeddings de posicion)
    "emb_dim": 768, # Embedding dimension
    "n_heads": 12, # Number of attention heads
    "n_layers": 12, # Number of layers
    "drop_rate": 0.1, # Dropout rate
    "qkv_bias": False # Query-Key-Value bias
}

### GPT Placeholder

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x

In [ ]:
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [ ]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Capas de Embeddinhs
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Capas de Transformers
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg)
            for _ in range(cfg["n_layers"])]
        )

        # Capa de normalizacion
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])

        # Capa de Output
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias = False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x) # Question: ¿Por que hace esto?
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        logits = self.out_head(x) # Question: ¿Como defino LOGITS?

        return logits



### Prepare Input data

In [ ]:
import tiktoken

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
batch = []

In [ ]:
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

In [ ]:
batch.append(torch.tensor(tokenizer.encode(txt1)))

In [ ]:
tokenizer.encode(txt1)

[6109, 3626, 6100, 345]

In [ ]:
torch.tensor(tokenizer.encode(txt1))

tensor([6109, 3626, 6100,  345])

In [ ]:
batch.append(torch.tensor(tokenizer.encode(txt2)))

In [ ]:
batch

[tensor([6109, 3626, 6100,  345]), tensor([6109, 1110, 6622,  257])]

In [ ]:
batch = torch.stack(batch, dim=0)

In [ ]:
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [ ]:
# Initialize a new GPT model

In [ ]:
torch.manual_seed(123)

In [ ]:
model = DummyGPTModel(GPT_CONFIG_124M)

In [ ]:
logits = model(batch)

In [ ]:
print(logits)

tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


### 4.2 Normalizing activations with layer normalization

La idea con esto es basicamente que los outputs de la red neuronal tengan media 0 y varianza 1.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn. Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)

In [ ]:
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())

In [ ]:
out = layer(batch_example)
print(out)

tensor([[0.0000, 0.4120, 0.0000, 0.1644, 0.0000, 0.6309],
        [0.0000, 1.0274, 0.6265, 0.8528, 0.2201, 0.2337]],
       grad_fn=<ReluBackward0>)


In [ ]:
# Before mean and variance

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

print("Mean: \n", mean)
print("Variance: \n", var)

Mean: 
 tensor([[0.2012],
        [0.4934]], grad_fn=<MeanBackward1>)
Variance: 
 tensor([[0.0704],
        [0.1635]], grad_fn=<VarBackward0>)


In [ ]:
# After layer normalization

In [ ]:
ln = LayerNorm(emb_dim=5)

In [ ]:
torch.set_printoptions(sci_mode=False) # Para leer mejor los valores cercanos a 0

In [ ]:
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean: \n", mean)
print("Variance: \n", var)

Mean: 
 tensor([[-0.0000],
        [ 0.0000]], grad_fn=<MeanBackward1>)
Variance: 
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


**Nota:**
* Diferencia entre `Batch normalization` y `"Layer" Normalization`.
  * `Batch normalization`, realiza la normalizacion en la dimension del batch mientras que el `Layer normalization` realiza la normalizacion en la dimension de la **feature**.

#### 4.3 Implementing a feed forward network with GELU (`Gaussian error linear unit`) activations

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

#### 4.4 Adding shortcut connections

Tambien se le conoce como skip o residual connetions(ResNet).

Esto se propueso para contrarestar el challenge de "vanishing gradientes".

Lo que pasa aqui es que los gradientes progresivamente se vuelven mas pequeños mientras se propagan por las calas, lo que lo hace mas dificil entrenar a las capas (layers)

Entonces este shortcut connection lo que hace es pasar el gradiente direcatmente hacia la siguiente capa haciendole skip. Es decir el output de una capa adicionandoselo directamente al output de la siguiente capa.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4* cfg["emb_dim"]),
            GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self, x):
        return self.layers(x)


#### 4.5 Connecting `attention` and `linear layers` in a `transformer block`

Ahora si a combinar `multi-head attention` , `layer normalization`, `dropout`, `feed forward layers` y `GELU` activations.

In [ ]:
cfg = GPT_CONFIG_124M

In [ ]:
cfg['emb_dim']

768

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):

        # Primera parte de Atention
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # add the original input back

        # Segunda parte de Feedforward
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [ ]:
torch.manual_seed(123)

In [ ]:
x = torch.rand(2,4,768)

In [ ]:
x

tensor([[[0.2961, 0.5166, 0.2517,  ..., 0.9541, 0.8567, 0.4604],
         [0.2238, 0.3047, 0.3019,  ..., 0.5465, 0.4532, 0.7598],
         [0.6945, 0.2478, 0.4111,  ..., 0.8838, 0.4898, 0.5963],
         [0.0890, 0.7804, 0.9223,  ..., 0.4507, 0.6357, 0.5833]],

        [[0.5716, 0.9297, 0.3396,  ..., 0.0477, 0.4564, 0.2797],
         [0.0936, 0.2211, 0.3806,  ..., 0.3948, 0.4545, 0.4536],
         [0.6788, 0.1741, 0.2084,  ..., 0.5557, 0.5930, 0.0959],
         [0.3894, 0.4083, 0.0662,  ..., 0.9861, 0.9341, 0.1319]]])

In [ ]:
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

In [ ]:
output

tensor([[[-0.0055,  0.0972, -0.1122,  ...,  1.2889,  0.2623,  0.6685],
         [ 0.0023, -0.2369,  0.1720,  ...,  0.5952,  0.2497,  0.7447],
         [ 0.4673,  0.4472,  0.1791,  ...,  1.2525,  0.3045,  0.7750],
         [ 0.0662,  0.7224,  0.9206,  ...,  0.4790,  0.7428,  0.7015]],

        [[ 0.3622,  1.2144,  0.5221,  ...,  0.1854,  0.0111, -0.5034],
         [-0.0225,  0.7789,  0.2770,  ...,  0.1734,  0.5419,  0.1143],
         [ 0.7425,  0.4013,  0.3211,  ...,  0.3268,  0.7523, -0.1642],
         [ 0.5745,  0.6241,  0.4410,  ...,  1.1963,  1.2650,  0.2243]]],
       grad_fn=<AddBackward0>)

### 4.6 Coding the `GPT model`

Assemble a fully working version of the original `124-million parameter` version of `GPT-2`.

Por ejemplo, en el caso de GPT-2 model, esta repetido 12 veces.

**Nota:** El output del transformer block lo pasamos por una capa de normalizacion ANTES de la capa lineal de output.

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Transformers Block Definition
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        # Token Embeddings
        tok_embeds = self.tok_emb(in_idx)

        # Positional Embeddings
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        # Sumamos los dos embedidng
        x = tok_embeds + pos_embeds

        # Aplico un Dropout
        x = self.drop_emb(x)

        # Aplico Bloques de Transformers
        x = self.trf_blocks(x)

        # Ultima capa de Normalizacion
        x = self.final_norm(x)

        logits = self.out_head(x)

        return logits


In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

In [ ]:
batch

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

In [ ]:
out = model(batch)

In [ ]:
out

tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)

Ok mira esta tan interesante: El input son los tokens que estoy transformando:

In [ ]:
print("Input batch:\n", batch)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


El output son de los dos rows del batch:

Nota interesante: Mirar que ya esta con la dimension determinada!.

In [ ]:
print("\nOutput shape:", out.shape)


Output shape: torch.Size([2, 4, 50257])


Como le pasamos 2 input text con 4 tokens cada uno, podemos ver que ya tenemos 2 salidas con las dimensiones del diccionario.

In [ ]:
# Number total of parameters of the model

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

In [ ]:
total_params

163009536

#### 4.7 Generating Text

Ahora viene la parte donde tomo el Output del modelo de GPT de nuevo en texto.

In [ ]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded: ", encoded)

# Lo transformo en tesor.
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

encoded:  [15496, 11, 314, 716]


In [ ]:
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feature

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=-1)

    return idx

In [ ]:
#### TODO: DEBUGGEAR LINEA POR LINEA generate_text_simple

In [ ]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)

print("encoded:", encoded)

encoded: [15496, 11, 314, 716]


In [ ]:
encoded_tensor = torch.tensor(encoded).unsqueeze(0) # Adds batch dimension

In [ ]:
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded_tensor.shape: torch.Size([1, 4])


In [ ]:
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
Output length: 10


Ahora usaremos el .decode del tokenizer para convertir los IDs de nuevo a texto:

In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())

In [ ]:
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue
